# **Notebook 4b: Creating weak labels and using precision filter for quality selection**

## 1. Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import precision_score, roc_auc_score

## 2. Preparation

In [ ]:
# Paths
FEATURES_TRAIN     = "./Data_with_features/features_train.csv"
FEATURES_VAL       = "./Data_with_features/features_val.csv"
FEATURES_UNLABELED = "./Data_with_features/features_unlabeled_sample.csv"

MODEL_DIR      = "./model_labeling"
LABEL_MODEL_PATH  = os.path.join(MODEL_DIR, "lr1_labeling_model.joblib")
LABEL_SCALER_PATH = os.path.join(MODEL_DIR, "lr1_scaler.joblib")
FEATURE_COLS_PATH = os.path.join(MODEL_DIR, "lr1_feature_cols.json")

WEAK_OUT = "data_weak_labeled95.csv"

# Columns
LABEL_COL = "unnecessary"
ID_COLS   = ["blurb_id", "blurb_sent"]
EXCLUDE   = ID_COLS + [LABEL_COL]

## 3. Load labeling model + scaler and reconstruct the exact column set

In [ ]:
# Load feature columns saved in 3b
with open(FEATURE_COLS_PATH) as f:
    lr1_cols = json.load(f)
print(f"Label model feature count: {len(lr1_cols)}")

# Load model and scaler
lr1_model  = joblib.load(LABEL_MODEL_PATH)
lr1_scaler = joblib.load(LABEL_SCALER_PATH)

# Verify column count matches scaler
expected = lr1_scaler.n_features_in_
assert len(lr1_cols) == expected, (
    f"Column-count mismatch: {len(lr1_cols)} cols but scaler expects {expected}.")

# Compute val probabilities for threshold tuning
df_val = pd.read_csv(FEATURES_VAL)
X_val  = df_val[lr1_cols].copy()
y_val  = df_val[LABEL_COL].astype(int).values
val_probs = lr1_model.predict_proba(lr1_scaler.transform(X_val))[:, 1]

print(f"Val AUC: {roc_auc_score(y_val, val_probs):.4f}")

Label model feature count: 57
Val AUC: 0.8704


## 4. Precision-coverage tradeoff across candidate precision targets

In [ ]:
# Computing precision on class 1 (unnecessary) for sentences where prob >= t,
# returns (precision, number of sentences selected)
def precision_remove_side(probs, labels, t):
    mask = probs >= t
    n = mask.sum()
    if n == 0:
        return np.nan, 0
    return precision_score(labels[mask], np.ones(n, dtype=int), pos_label=1, zero_division=0), n

# Computing precision on class 0 (necessary) for sentences where prob <= t,
# returns (precision, number of sentences selected)
def precision_keep_side(probs, labels, t):
    mask = probs <= t
    n = mask.sum()
    if n == 0:
        return np.nan, 0
    return precision_score(labels[mask], np.zeros(n, dtype=int), pos_label=0, zero_division=0), n

# Threshold grid to sweep over
grid = np.round(np.arange(0.01, 1.00, 0.01), 2)

# Minimum sentences required at a threshold before trusting its precision estimate
MIN_SUPPORT = 10

# Showing precision and coverage tradeoff at different precision targets
targets = [0.80, 0.85, 0.90, 0.95]
rows = []

for target in targets:
    # Finding most permissive threshold per class that still meets precision target
    t_unnec = next((t for t in grid
                    if precision_remove_side(val_probs, y_val, t)[1] >= MIN_SUPPORT
                    and precision_remove_side(val_probs, y_val, t)[0] >= target), None)

    t_nec = next((t for t in grid[::-1]
                  if precision_keep_side(val_probs, y_val, t)[1] >= MIN_SUPPORT
                  and precision_keep_side(val_probs, y_val, t)[0] >= target), None)

    coverage = 0
    if t_unnec and t_nec:
        coverage = ((val_probs >= t_unnec) | (val_probs <= t_nec)).sum() / len(val_probs)

    rows.append({"target_precision": target, "T_unnec": t_unnec,
                 "T_nec": t_nec, "val_coverage": coverage})

coverage_df = pd.DataFrame(rows)
print(coverage_df.to_string(index=False))

 target_precision  T_unnec  T_nec  val_coverage
             0.80     0.93   0.92      0.980132
             0.85     0.95   0.81      0.877483
             0.90     0.95   0.64      0.771523
             0.95     0.96   0.31      0.559603


In [ ]:
TARGET_PRECISION_REMOVE = 0.95
TARGET_PRECISION_KEEP   = 0.95

## 5. Selecting final thresholds based on chosen precision target

In [ ]:
treshold_unnec = next((t for t in grid
                       if precision_remove_side(val_probs, y_val, t)[1] >= MIN_SUPPORT
                       and precision_remove_side(val_probs, y_val, t)[0] >= TARGET_PRECISION_REMOVE), None)

treshold_nec = next((t for t in grid[::-1]
                     if precision_keep_side(val_probs, y_val, t)[1] >= MIN_SUPPORT
                     and precision_keep_side(val_probs, y_val, t)[0] >= TARGET_PRECISION_KEEP), None)

print(f"T_unnecessary : {treshold_unnec}")
print(f"T_necessary   : {treshold_nec}")

T_unnecessary : 0.96
T_necessary   : 0.31


## 7. Apply labeling model to the unlabeled data sample (25K)

In [ ]:
# Loading unlabeled features
df_unlab = pd.read_csv(FEATURES_UNLABELED)
print(f"Unlabeled: {len(df_unlab)} sentences")

# Verifying all required features are present
missing = [c for c in lr1_cols if c not in df_unlab.columns]
assert not missing, f"Missing columns: {missing}"

# Scaling features and compute probabilities
X_unlab     = df_unlab[lr1_cols].copy()
unlab_probs = lr1_model.predict_proba(lr1_scaler.transform(X_unlab))[:, 1]

# Assigning weak labels: 1 if prob >= T_unnecessary, 0 if prob <= T_necessary, else abstain
weak = np.full(len(unlab_probs), np.nan)
weak[unlab_probs >= treshold_unnec] = 1
weak[unlab_probs <= treshold_nec]   = 0

n_abstain = np.isnan(weak).sum()
print(f"Abstained:    {n_abstain} ({n_abstain/len(weak):.1%})")
print(f"Weak-labeled: {len(weak) - n_abstain} ({1 - n_abstain/len(weak):.1%})")

Unlabeled: 25000 sentences
Abstained:    16904 (67.6%)
Weak-labeled: 8096 (32.4%)


## 8. Building and saving the weak-label file (abstentions dropped)

In [ ]:
# Building output dataframe with ID columns, probability and weak label
out = df_unlab[ID_COLS + (["blurb"] if "blurb" in df_unlab.columns else [])].copy()
out["prob_unnecessary"] = unlab_probs
out[LABEL_COL] = weak

# Dropping abstained sentences (only keep confident predictions)
out_confident = out[out[LABEL_COL].notna()].copy()
out_confident[LABEL_COL] = out_confident[LABEL_COL].astype(int)

# Saving weak labels to csv
out_confident.to_csv(WEAK_OUT, index=False)
print(f"Saved {len(out_confident)} weak labels -> {WEAK_OUT}")

# Printing label distribution
print("\nWeak label distribution (0=necessary, 1=unnecessary):")
print(out_confident[LABEL_COL].value_counts().to_string())
print(out_confident[LABEL_COL].value_counts(normalize=True).round(3).to_string())

# Saving thresholds for later use in label quality evaluation notebook
with open(os.path.join(MODEL_DIR, "lr1_thresholds.json"), "w") as f:
    json.dump({
        "T_unnecessary": None if treshold_unnec is None else float(treshold_unnec),
        "T_necessary":   None if treshold_nec   is None else float(treshold_nec),
        "target_precision": TARGET_PRECISION_REMOVE,
    }, f, indent=2)
print("Saved thresholds -> lr1_thresholds.json")

Saved 8096 weak labels -> data_weak_labeled95.csv

Weak label distribution (0=necessary, 1=unnecessary):
unnecessary
0    4817
1    3279
unnecessary
0    0.595
1    0.405
Saved thresholds -> lr1_thresholds.json


## 9. Summary of the process

In [ ]:
n_total = len(unlab_probs)
n_keep  = int((out_confident[LABEL_COL] == 0).sum())
n_remove = int((out_confident[LABEL_COL] == 1).sum())
print("Pseudo-labeling summary")
print(f"  Unlabeled sentences:      {n_total}")
print(f"  Emitted pseudo-labels:    {n_keep + n_remove} ({(n_keep+n_remove)/n_total:.1%} coverage)")
print(f"    -> keep (0):            {n_keep}")
print(f"    -> remove (1):          {n_remove}")
print(f"  Abstained:                {n_total - n_keep - n_remove}")
print(f"  treshold_necessary={treshold_nec}  treshold_unnecessary={treshold_unnec}")

Pseudo-labeling summary
  Unlabeled sentences:      25000
  Emitted pseudo-labels:    8096 (32.4% coverage)
    -> keep (0):            4817
    -> remove (1):          3279
  Abstained:                16904
  treshold_necessary=0.31  treshold_unnecessary=0.96
